# UniCompress — Python quickstart

`unicompress` is a Cython extension over the UniCompress C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install unicompress
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## Two layers

`deflate`/`inflate` are the raw RFC 1951 bitstream. `zlib_deflate`/`zlib_inflate`
are the RFC 1950 framing around it: two header bytes, the body, and an Adler-32
of the original data.

In [1]:
import unicompress

payload = b"deflate " * 200
raw = unicompress.deflate(payload)
stream = unicompress.zlib_deflate(payload)
{
    "input": len(payload),
    "raw deflate": len(raw),
    "zlib stream": len(stream),
    "framing overhead": len(stream) - len(raw),
}

{'input': 1600, 'raw deflate': 22, 'zlib stream': 28, 'framing overhead': 6}

## The standard library agrees

`zlib` reads what this writes and writes what this reads, in both directions and
at both layers — `wbits=-15` selects the raw bitstream, the default the
container.

In [2]:
import zlib

(zlib.decompress(raw, wbits=-15) == payload,
 zlib.decompress(stream) == payload,
 unicompress.inflate(zlib.compress(payload)[2:-4]) == payload,
 unicompress.zlib_inflate(zlib.compress(payload)) == payload)

(True, True, True, True)

## Decompression is bounded

A small stream can describe a very large one. Every decode takes a ceiling and
refuses rather than allocating past it.

In [3]:
bomb = unicompress.zlib_deflate(bytes(200_000))
print("compressed to", len(bomb), "bytes")
try:
    unicompress.zlib_inflate(bomb, max_output=1024)
except ValueError as exc:
    print("refused:", exc)

compressed to 1270 bytes
refused: codec operation failed


A corrupted trailer fails the Adler-32 check rather than returning
plausible-looking output.

In [4]:
damaged = bytearray(stream)
damaged[-1] ^= 0xFF
try:
    unicompress.zlib_inflate(bytes(damaged))
except ValueError as exc:
    print("refused:", exc)

refused: codec operation failed
